# Real-Time British Sign Language (BSL) Translation System
## Live Webcam Demonstration

---

**Project:** Real-Time BSL Translation System for Enhanced Communication

**Purpose:** Load a trained PyTorch EfficientNet-B0 model and recognise BSL signs from a live webcam feed, displaying the predicted English label in real time.

> **Important:** This notebook is for **inference only**. It does NOT retrain the model. It loads the saved checkpoint and uses it for real-time prediction.

**Prerequisites:**
- Trained model saved as `models/best_model.pth` or `models/final_model.pth`
- Class mapping saved as `annotations/class_mapping.json`
- A working webcam
- Recommended: Run locally in VS Code or Jupyter (not Colab) for best webcam support

## Section 1: Setup and Imports

Install the required packages (run once):
```bash
pip install torch torchvision opencv-python mediapipe Pillow numpy
```

In [ ]:
# Step 1: Import all required libraries

import os
import sys
import json
import time
import warnings
from pathlib import Path
from collections import deque

import numpy as np
import cv2
from PIL import Image

import torch
import torch.nn as nn
from torchvision import transforms, models

import mediapipe as mp

warnings.filterwarnings("ignore")

print("All libraries imported successfully.")

In [ ]:
# Step 2: Set device (CUDA if available, otherwise CPU)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem = torch.cuda.get_device_properties(0).total_mem / (1024**3)
    print(f"CUDA available! GPU: {gpu_name}, VRAM: {gpu_mem:.1f} GB")
else:
    print("CUDA not available. Using CPU (inference will be slower).")
print(f"Device: {DEVICE}")

## Section 2: Load Trained Model

In [ ]:
# Step 3: Define paths

MODEL_PATH = "models/final_model.pth"
ALTERNATIVE_MODEL = "models/best_model.pth"
CLASS_MAPPING_PATH = "annotations/class_mapping.json"

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]
DEFAULT_IMG_SIZE = 224

In [ ]:
# Step 4: Load class names
# Try checkpoint first, then fall back to JSON file

idx_to_class = None
num_classes = None
img_size = DEFAULT_IMG_SIZE

# Check if model file exists (try both paths)
model_file = MODEL_PATH if Path(MODEL_PATH).exists() else None
if model_file is None and Path(ALTERNATIVE_MODEL).exists():
    model_file = ALTERNATIVE_MODEL
    print(f"Using alternative model: {ALTERNATIVE_MODEL}")

if model_file:
    ckpt = torch.load(model_file, map_location=DEVICE, weights_only=False)

    # Extract class info from checkpoint if available
    if "reverse_mapping" in ckpt:
        raw = ckpt["reverse_mapping"]
        idx_to_class = {int(k): v for k, v in raw.items()}
    elif "idx_to_class" in ckpt:
        raw = ckpt["idx_to_class"]
        idx_to_class = {int(k): v for k, v in raw.items()}
    elif "class_mapping" in ckpt:
        raw = ckpt["class_mapping"]
        idx_to_class = {v: k for k, v in raw.items()}
    elif "class_names" in ckpt:
        idx_to_class = {i: n for i, n in enumerate(ckpt["class_names"])}

    num_classes = ckpt.get("num_classes")
    img_size = ckpt.get("img_size", ckpt.get("image_size", DEFAULT_IMG_SIZE))

# Fall back to JSON file if checkpoint had no class info
if idx_to_class is None:
    if Path(CLASS_MAPPING_PATH).exists():
        with open(CLASS_MAPPING_PATH, "r", encoding="utf-8") as f:
            data = json.load(f)
        if "reverse_mapping" in data:
            idx_to_class = {int(k): v for k, v in data["reverse_mapping"].items()}
        elif "class_mapping" in data:
            idx_to_class = {v: k for k, v in data["class_mapping"].items()}
        else:
            idx_to_class = {v: k for k, v in data.items()}
        print(f"Loaded class mapping from: {CLASS_MAPPING_PATH}")
    else:
        raise FileNotFoundError(f"No class mapping found. Check {CLASS_MAPPING_PATH}")

if num_classes is None:
    num_classes = len(idx_to_class)

print(f"Classes ({num_classes}): {list(idx_to_class.values())}")
print(f"Image size: {img_size}x{img_size}")

In [ ]:
# Step 5: Rebuild EfficientNet-B0 and load weights

def create_efficientnet_b0(num_classes):
    """Recreate EfficientNet-B0 matching the training architecture."""
    model = models.efficientnet_b0(weights=None)
    in_features = model.classifier[1].in_features
    model.classifier[1] = nn.Linear(in_features, num_classes)
    return model

# Build and load
model = create_efficientnet_b0(num_classes)

# Load state dict from checkpoint
state_dict = ckpt.get("model_state_dict", ckpt)
model.load_state_dict(state_dict, strict=False)

# Move to device and set to eval mode
model = model.to(DEVICE)
model.eval()

param_count = sum(p.numel() for p in model.parameters())
print(f"Model loaded: {num_classes} classes, {param_count:,} parameters, device={DEVICE}")

## Section 3: Define Preprocessing Pipeline

In [ ]:
# Step 6: Define image preprocessing (must match training)

transform = transforms.Compose([
    transforms.Resize((img_size, img_size)),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

def preprocess_frame(frame_bgr):
    """Convert OpenCV BGR frame to preprocessed tensor."""
    rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
    pil_img = Image.fromarray(rgb)
    tensor = transform(pil_img).unsqueeze(0)  # Add batch dim
    return tensor.to(DEVICE)

print(f"Transform pipeline ready: {img_size}x{img_size}, ImageNet normalization")

## Section 4: MediaPipe Hand Detection

In [ ]:
# Step 7: Initialise MediaPipe Hands

hands_detector = mp.solutions.hands.Hands(
    static_image_mode=False,
    max_num_hands=1,
    min_detection_confidence=0.5,
    min_tracking_confidence=0.5,
)

mp_drawing = mp.solutions.drawing_utils
mp_styles = mp.solutions.drawing_styles

print("MediaPipe Hands initialised (max 1 hand, confidence >= 0.5)")

In [ ]:
# Step 8: Hand detection and cropping function

BBOX_PADDING = 30
MIN_BBOX_SIZE = 40

def detect_and_crop_hand(frame_bgr, hands_det):
    """Detect hand and return crop + bounding box."""
    h, w = frame_bgr.shape[:2]
    rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
    results = hands_det.process(rgb)

    if not results.multi_hand_landmarks:
        return None, None, False, results

    hand_lm = results.multi_hand_landmarks[0]
    xc = [lm.x * w for lm in hand_lm.landmark]
    yc = [lm.y * h for lm in hand_lm.landmark]

    x1 = max(0, int(min(xc)) - BBOX_PADDING)
    y1 = max(0, int(min(yc)) - BBOX_PADDING)
    x2 = min(w, int(max(xc)) + BBOX_PADDING)
    y2 = min(h, int(max(yc)) + BBOX_PADDING)

    if (x2 - x1) < MIN_BBOX_SIZE or (y2 - y1) < MIN_BBOX_SIZE:
        return None, (x1, y1, x2, y2), False, results

    crop = frame_bgr[y1:y2, x1:x2]
    if crop.size == 0:
        return None, (x1, y1, x2, y2), False, results

    return crop, (x1, y1, x2, y2), True, results

## Section 5: Prediction Function

In [ ]:
# Step 9: Prediction function

def predict_sign(cropped_hand):
    """Run model inference on a cropped hand image."""
    input_tensor = preprocess_frame(cropped_hand)
    with torch.inference_mode():
        outputs = model(input_tensor)
    probs = torch.softmax(outputs, dim=1)
    confidence, pred_idx = probs.max(dim=1)
    class_name = idx_to_class.get(pred_idx.item(), f"Class_{pred_idx.item()}")
    return class_name, confidence.item()

In [ ]:
# Steps 10-11: Confidence threshold and prediction smoothing

CONFIDENCE_THRESHOLD = 0.80
SMOOTHING_WINDOW = 10

class PredictionSmoother:
    """Majority-vote smoothing over a sliding window."""
    def __init__(self, window=SMOOTHING_WINDOW):
        self.buf = deque(maxlen=window)

    def add(self, label, conf):
        self.buf.append(label if conf >= CONFIDENCE_THRESHOLD else "Uncertain")

    def get(self):
        if not self.buf: return "No data", False
        counts = {}
        for l in self.buf: counts[l] = counts.get(l, 0) + 1
        best = max(counts, key=counts.get)
        return best, counts[best] == len(self.buf)

smoother = PredictionSmoother()
print(f"Confidence threshold: {CONFIDENCE_THRESHOLD:.0%}, Smoothing: {SMOOTHING_WINDOW} frames")

## Section 6: Real-Time Webcam Demo

> **Note for Google Colab users:** Direct webcam access in Colab is limited. For the best experience, **run this notebook locally** in VS Code, Jupyter Notebook, or as a standalone Python script. If using Colab, you can use Colab's built-in camera feature, but the frame rate and display will be lower.

In [ ]:
# Step 12-13: FPS counter and main webcam loop

class FPSCounter:
    def __init__(self, window=30):
        self.ts = deque(maxlen=window)
    def tick(self):
        now = time.time(); self.ts.append(now)
        if len(self.ts) < 2: return 0.0
        return (len(self.ts) - 1) / (self.ts[-1] - self.ts[0])


def draw_overlay(frame, label, conf, stable, fps, smooth_n):
    """Draw info panel and bounding box on the frame."""
    h, w = frame.shape[:2]
    # Top panel background
    overlay = frame.copy()
    cv2.rectangle(overlay, (0, 0), (w, 90), (30, 30, 30), -1)
    cv2.addWeighted(overlay, 0.75, frame, 0.25, 0, frame)

    # Colour coding
    if label == "No hand detected": colour = (100, 100, 100)
    elif label == "Uncertain": colour = (0, 165, 255)
    elif stable: colour = (0, 255, 100)
    else: colour = (50, 200, 255)

    cv2.putText(frame, f"BSL Sign:  {label}", (20, 38),
                cv2.FONT_HERSHEY_SIMPLEX, 0.95, colour, 2, cv2.LINE_AA)

    status = " (Stable)" if stable else ""
    cv2.putText(frame, f"Confidence: {conf:.1%}{status}", (20, 68),
                cv2.FONT_HERSHEY_SIMPLEX, 0.55, (200, 200, 200), 1, cv2.LINE_AA)
    cv2.putText(frame, f"FPS: {fps:.1f}", (w - 140, 38),
                cv2.FONT_HERSHEY_SIMPLEX, 0.65, (0, 255, 255), 2, cv2.LINE_AA)
    cv2.putText(frame, f"Smooth: {smooth_n}/{SMOOTHING_WINDOW}", (w - 200, 68),
                cv2.FONT_HERSHEY_SIMPLEX, 0.5, (150, 150, 150), 1, cv2.LINE_AA)

    # Bottom bar
    cv2.rectangle(frame, (0, h - 30), (w, h), (30, 30, 30), -1)
    cv2.putText(frame, "Press 'q' to quit  |  Show your hand to the camera", (20, h - 10),
                cv2.FONT_HERSHEY_SIMPLEX, 0.5, (180, 180, 180), 1, cv2.LINE_AA)
    return frame, colour

In [ ]:
# Step 13: Main webcam loop

def run_demo(webcam_idx=0):
    """Run the real-time BSL sign recognition demo."""
    cap = cv2.VideoCapture(webcam_idx)
    if not cap.isOpened():
        raise RuntimeError("Cannot open webcam. Check connection.")

    cap.set(cv2.CAP_PROP_FRAME_WIDTH, 900)
    cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 700)

    smoother_demo = PredictionSmoother()
    fps_ctr = FPSCounter()
    total_frames = 0; total_infer = 0; infer_times = []
    t_start = time.time()

    print("\n" + "=" * 60)
    print("  REAL-TIME BSL SIGN RECOGNITION")
    print("  Press 'q' to quit")
    print("=" * 60 + "\n")

    while True:
        ret, frame = cap.read()
        if not ret: continue
        total_frames += 1
        frame = cv2.flip(frame, 1)  # Mirror

        # Detect hand
        crop, bbox, detected, mp_res = detect_and_crop_hand(frame, hands_detector)

        label = "No hand detected"; conf = 0.0; stable = False; colour = (100,100,100)

        if detected and crop is not None:
            t0 = time.time()
            pred_name, pred_conf = predict_sign(crop)
            infer_times.append(time.time() - t0)
            total_infer += 1

            # Confidence threshold
            if pred_conf >= CONFIDENCE_THRESHOLD:
                label = pred_name
            else:
                label = "Uncertain"
            conf = pred_conf

            # Smoothing
            smoother_demo.add(label, conf)
            label, stable = smoother_demo.get()

        # Draw landmarks
        if mp_res and mp_res.multi_hand_landmarks:
            for hlm in mp_res.multi_hand_landmarks:
                mp_drawing.draw_landmarks(
                    frame, hlm, mp.solutions.hands.HAND_CONNECTIONS,
                    mp_styles.get_default_hand_landmarks_style(),
                    mp_styles.get_default_hand_connections_style())

        # Draw bounding box with corner accents
        if bbox:
            x1, y1, x2, y2 = bbox
            _, colour = draw_overlay(frame, label, conf, stable, 0, 0)
            cv2.rectangle(frame, (x1, y1), (x2, y2), colour, 2)
            cl = 15
            for cx, cy, dx, dy in [(x1,y1,1,1),(x2,y1,-1,1),(x1,y2,1,-1),(x2,y2,-1,-1)]:
                cv2.line(frame, (cx, cy), (cx + dx*cl, cy), colour, 3)
                cv2.line(frame, (cx, cy), (cx, cy + dy*cl), colour, 3)

        # Draw info panel
        frame, _ = draw_overlay(frame, label, conf, stable, fps_ctr.tick(), len(smoother_demo.buf))

        cv2.imshow("BSL Sign Recognition - Real-Time Demo", frame)
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

    # Step 14: Release resources
    cap.release()
    cv2.destroyAllWindows()

    duration = time.time() - t_start
    avg_fps = total_frames / duration if duration else 0
    avg_inf = np.mean(infer_times) * 1000 if infer_times else 0

    print("\n" + "=" * 60)
    print("  SESSION SUMMARY")
    print("=" * 60)
    print(f"  Total frames: {total_frames}")
    print(f"  Inferences: {total_infer}")
    print(f"  Avg FPS: {avg_fps:.1f}")
    print(f"  Avg inference: {avg_inf:.2f} ms")
    print(f"  Duration: {duration:.1f}s")
    print("=" * 60)

    return {
        "total_frames": total_frames, "inferences": total_infer,
        "avg_fps": round(avg_fps, 1), "avg_inference_ms": round(avg_inf, 2),
        "duration_s": round(duration, 1), "device": str(DEVICE),
    }

In [ ]:
# Run the demo!
# Change webcam_idx if your camera is not at index 0

try:
    perf = run_demo(webcam_idx=0)
except RuntimeError as e:
    print(f"ERROR: {e}")
    print("Make sure your webcam is connected and not in use.")
except KeyboardInterrupt:
    print("\nDemo interrupted.")
    cv2.destroyAllWindows()

## Section 7: Troubleshooting

### Common Issues and Solutions:

| Problem | Solution |
|---------|----------|
| `Model file not found` | Ensure `models/final_model.pth` or `models/best_model.pth` exists |
| `Class mapping not found` | Ensure `annotations/class_mapping.json` exists |
| `Cannot open webcam` | Check camera connection; try changing `webcam_idx` to 1 or 2 |
| `No hand detected` | Ensure good lighting and your hand is clearly visible |
| `CUDA out of memory` | Close other GPU applications; the demo uses very little VRAM |
| `Slow predictions` | Ensure GPU is enabled; check `torch.cuda.is_available()` |
| `Wrong predictions` | Verify the model matches the class mapping; check preprocessing |

### Google Colab Note:
- Direct `cv2.imshow()` does NOT work in Colab
- Use `google.colab.patches` for camera capture as an alternative
- For best results, download this notebook and run locally

## Section 8: Presentation Demo Explanation

**Use this explanation during your presentation:**

> This demo uses a webcam to capture the signer's hand in real time. MediaPipe, a state-of-the-art hand tracking library developed by Google, detects the hand and extracts 21 landmark points on the hand structure. From these landmarks, we compute a bounding box around the hand and crop just the hand region from the full camera frame.
>
> This crop is then preprocessed, resized to 224 by 224 pixels, normalised using ImageNet statistics, and fed into our trained EfficientNet-B0 convolutional neural network. The model, which was trained on the BSL Fingerspelling Dataset containing over 20,000 labelled images across 26 letter classes, outputs a probability distribution over all possible BSL signs.
>
> A confidence threshold of 80 percent is applied so that only high-confidence predictions are displayed. Additionally, a majority-vote smoothing mechanism over the last 10 frames ensures stable, flicker-free predictions on screen. The predicted BSL letter is displayed as English text overlay on the webcam feed in real time.
>
> The entire inference pipeline runs at approximately 15 to 30 frames per second on a GPU, making it suitable for practical, real-time communication assistance.

## Section 9: Limitations

### Known Limitations of This System:

1. **Lighting dependency:** Works best in well-lit environments. Poor lighting degrades MediaPipe hand detection and model accuracy significantly.

2. **Hand visibility:** The signing hand must be clearly visible and unoccluded. Wearing gloves, long sleeves, or holding objects may prevent detection.

3. **Limited vocabulary:** The system recognises only the signs it was trained on (26 BSL alphabet letters in this case). It does not recognise numbers, common words, or full phrases.

4. **Visually similar signs:** Signs with similar hand shapes (such as U/V, M/N, or D/G) may be confused by the model, especially at certain angles.

5. **No BSL grammar translation:** This system performs isolated sign recognition only. It does not translate BSL grammar, sentence structure, facial expressions, or body posture, which are essential components of full BSL communication.

6. **Camera quality dependency:** Recognition accuracy varies with camera resolution, frame rate, colour accuracy, and lens quality.

7. **Static image model:** The model was trained on static images, not video sequences. Signs that rely on motion or dynamic gestures cannot be recognised without temporal modelling (CNN-LSTM or 3D CNN).

8. **Generalisation:** The model may not generalise well to hand shapes, skin tones, or signing styles not well represented in the training data.